# Research Discussion #1



## Commercial: Personalized Recommender

X.com's recommender system is a neat one to discuss because it's a hybrid of traditional recommenders with transformers. They also have a [readme](https://github.com/xai-org/x-algorithm).

The recommender system decides what to put on your "For You" page on the X app, which should be posts that you should like and overtime it should be curated to your viewing preferences. It uses *Thunder* and *Phoenix*, which are models that when combined gives the orchestration pipeline the data it needs to then be filtered down further. The cool thing is that *Phoenix* is transformer based, using Grok to determine the rankings of posts that you might like to see on your "For You" page. 

**Home Mixer**
The orchestration model, where Thunder and Phoenix are used to feed data into the model. The entire model:

1. **Query Hydration**
   - Load user context
   - Fetch recent activity, history, preferences, and request metadata

2. **Candidate Sources**
   - **Thunder**
     - Retrieves in-network posts from followed accounts
   - **Phoenix Retrieval**
     - Retrieves out-of-network candidates from the global corpus

3. **Candidate Hydration**
   - Add post metadata
   - Add author metadata
   - Add media, engagement, and safety/context features

4. **Filtering**
   - Remove invalid candidates
   - Remove blocked, muted, duplicate, stale, or already-seen posts
   - Apply policy and visibility filters

5. **Scoring**
   - **Phoenix Scorer**
     - Predicts engagement probabilities per candidate
   - **Weighted Scorer**
     - Combines predicted actions into a single relevance score
   - **Diversity / OON Scorers**
     - Adjust scores for author diversity and out-of-network content

6. **Top-K Selection**
   - Sort candidates by final adjusted score
   - Select the highest-ranking K candidates

7. **Ranked Feed Response**
   - Return the final ordered set of posts for the For You feed

**Thunder**
Finds posts based off the people you follow, which are considered to be *in-network*. We can assume that the user follows people they want to see posts from, so this model generates a list of posts from those users. 

**Phoenix**
Finds posts based off the people who are *out-of-network* and uses Grok to score the posts based off a ranking system, which is determined by the following: 

- Retrieval: Efficiently narrow down millions of candidates to hundreds using approximate nearest neighbor (ANN) search
- Ranking: Score and order the retrieved candidates using a more expressive transformer model

Phoenix makes the following calculations per post: 

**Predictions:**
```
├── P(favorite)
├── P(reply)
├── P(repost)
├── P(quote)
├── P(click)
├── P(profile_click)
├── P(video_view)
├── P(photo_expand)
├── P(share)
├── P(dwell)
├── P(follow_author)
├── P(not_interested)
├── P(block_author)
├── P(mute_author)
└── P(report)
```
Then creates a score using the following calculation: `Final Score = Σ (weight_i × P(action_i))`

This score is then calculated with the following:
- `Author Diversity Scorer`
    - Attenuate repeated author scores for diversity
- `OON Scorer`
    - Adjust scores for out-of-network content

These scores are then averaged as `K`. This then feeds into the other processes in the *Home Mixer* which basically adds metadata, sorts, and filters, with the final result the posts you would receive in your "For You" page. 

### Thoughts

I think it's really awesome, they are able to do all this at inference and make use of some interesting hashing methods to make it work extremely quick. It's really interesting to see the code of a recommender system that generates outputs live while millions of people are simultaneously using the app getting their own recommendations. It theoretically should deliver a good user experience, even if the person is not actively curating their own content. It should only get better with time as the transformer get's better on making recommendations. 

## Non-Personalized Recommenders: Metacritic

I decided to choose Metacritic because I use it the most out of the three recommenders. It is a pretty basic system: it gathers scores from trusted critic sources, applies different weights to those sources, and then calculates an overall average.

What makes it useful is not that the technique is complicated, but that the quality of the sources and the weighting system is why I like it over the other recommenders like Rotten Tomatoes and IMDB. 

All these recommenders sidestep the issue of review bombing, where user scores can be heavily influenced by controversy, politics, or people reacting emotionally to something offensive. Metacritic avoids some of that problem by keeping its main recommendation score based on selected critics rather than the general public. However, Metacritic seems to be more in line with audience scores when there's not any review bombing, opposed to others like Rotten Tomatoes which (anecdotally) has critics that aren't always in line with the audience.

## Article Discussion

A similar example is review bombing on sites like Rotten Tomatoes or IMDb. This happens when a large group of users gives something extremely low ratings before or right after release, which can make the recommendation score look worse than the actual audience reaction.

The article focused on *The Promise*, but this has happened in other cases too, with both negative and positive ratings. For example, some Harry Potter-related media has been review bombed because of political controversy around the author. The opposite can also happen, where fans “love bomb” something by giving it very high scores. An example is the anime *Frieren*, which became highly ranked partly because it had a very active and enthusiastic fanbase. Sometimes this is serious, and sometimes people do it just because they think it is funny to push something to the top.

Community ratings are usually not the main factor in a work’s official rating, but they can still make the work look bad to users. If people see a very low audience score, they may assume the movie, show, or game is bad before looking any deeper.

One way to handle this is to temporarily pause or limit the rating system when there is an unusual spike in low ratings. For example, if a movie suddenly gets thousands of one-star reviews from new accounts, the system could flag that activity and reduce its effect until it is reviewed.

There may not be a perfect solution, though. The best option is probably to build a stronger trust system for accounts. Ratings from verified viewers, older accounts, or users with normal rating patterns should count more than ratings from suspicious accounts. That would not stop review bombing completely, but it would make it harder for coordinated groups to manipulate recommendations.